# BiomedParse: one image, one P_ALL prompt -> multiple region masks

Runs BiomedParse (GPU) over the `P_ALL` rows of the `acdc`, `chaos`, and
`isic_task2` manifests (see `DATASETS.md`). Each `P_ALL` row names *all* regions
present in one image in a single natural-language prompt (`prompt_text`), and
carries the individual region names separately in `target_regions`
(semicolon-separated slugs) and their ground-truth masks in `mask_paths`.

BiomedParse's own inference API (`interactive_infer_image(model, image, prompts)`)
does not parse one free-text sentence into sub-targets -- it takes a **list** of
clinical noun-phrase text prompts against **one shared image encoding** and
returns **one probability mask per prompt** in that list, all in a single forward
pass. So this notebook turns each `P_ALL` row's `target_regions` into that list of
per-region prompts, calls BiomedParse once per image, and gets back a separate
mask for every region named in the prompt -- genuinely "one image, one prompt,
multiple masks," not a merged blob.

**Limited to 20 images per dataset** (first 20 `P_ALL` rows encountered in each
manifest -- this happens to be all of them, since each manifest has exactly 20)
-- see `LIMIT_PER_DATASET` below to change that. Datasets are run **one at a
time and in order** (ACDC, then CHAOS, then ISIC Task 2) -- each is its own
cell near the bottom, so you can watch one finish before the next starts
rather than firing all three at once.

**GPU (T4).** Make sure Runtime > Change runtime type > T4 GPU is selected
before running. BiomedParse's own inference code hardcodes `.cuda()` (there's
no official CPU path), so on a T4 this notebook runs it exactly as intended --
no patching needed. The next cell confirms a GPU is actually visible before
anything else runs.

**Dataset location: Google Drive.** This mounts your Drive and expects
`dataset/acdc`, `dataset/chaos`, `dataset/isic_task2` (same structure as this
repo's own `dataset/` folder, each with its `manifest_<name>.csv`) directly
under "My Drive". If your dataset folder is nested deeper (e.g. inside a
project folder), edit `DRIVE_SUBPATH` in the next-but-one cell.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > T4 GPU, then "
    "Runtime > Restart session, and re-run from the top."
)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

# Your dataset is on Google Drive with the same folder structure as this repo's
# dataset/ (dataset/acdc, dataset/chaos, dataset/isic_task2, each with its own
# manifest_<name>.csv). Edit DRIVE_SUBPATH below if your "dataset" folder sits
# somewhere other than directly under "My Drive".
DRIVE_SUBPATH = "dataset"  # e.g. "positional-bias-medical-vlms/dataset" if nested
DATASET_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBPATH

# Results are also written to Drive (not /content) so they survive a Colab
# disconnect/session reset instead of being lost with the runtime.
RESULTS_ROOT = Path("/content/drive/MyDrive/results/biomedparse")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = ["acdc", "chaos", "isic_task2"]
LIMIT_PER_DATASET = 20  # run the first 20 P_ALL rows (20 images) per dataset

for name in DATASETS:
    manifest = DATASET_ROOT / name / f"manifest_{name}.csv"
    assert manifest.exists(), f"Missing {manifest} -- fix DRIVE_SUBPATH above."
print("Dataset root OK:", DATASET_ROOT)
print("Results root:", RESULTS_ROOT)

In [ ]:
# Clone BiomedParse and install its dependencies. torch/torchvision are left
# alone (Colab already ships a working CUDA-enabled torch matched to its GPU
# driver) -- installing the repo's own pinned torch would risk breaking that.
![ -d BiomedParse ] || git clone -q https://github.com/microsoft/BiomedParse.git

req_path = "BiomedParse/assets/requirements/requirements.txt"
filtered_path = "/content/biomedparse_requirements_no_torch.txt"
with open(req_path) as f_in, open(filtered_path, "w") as f_out:
    for line in f_in:
        if line.strip().lower().startswith(("torch", "torchvision", "torchaudio")):
            continue
        f_out.write(line)

!pip install -q -r {filtered_path}

In [ ]:
import sys

BIOMEDPARSE_DIR = "/content/BiomedParse"
sys.path.insert(0, BIOMEDPARSE_DIR)

In [ ]:
import os
os.chdir(BIOMEDPARSE_DIR)  # BiomedParse's own configs/imports assume this cwd

import torch
from modeling.BaseModel import BaseModel
from modeling import build_model
from utilities.distributed import init_distributed
from utilities.arguments import load_opt_from_config_files
from utilities.constants import BIOMED_CLASSES
from inference_utils.inference import interactive_infer_image

opt = load_opt_from_config_files(["configs/biomedparse_inference.yaml"])
opt = init_distributed(opt)

model = BaseModel(opt, build_model(opt)).from_pretrained("hf_hub:microsoft/BiomedParse").eval().cuda()

with torch.no_grad():
    model.model.sem_seg_head.predictor.lang_encoder.get_text_embeddings(
        BIOMED_CLASSES + ["background"], is_eval=True
    )

print("BiomedParse loaded on GPU.")

In [ ]:
import csv
import time

import numpy as np
from PIL import Image

RESULT_FIELDS = [
    "dataset", "sample_id", "prompt_id", "region", "prompt_text",
    "image_path", "gt_mask_path", "pred_mask_path",
    "dice", "iou",
    "gt_centroid_x", "gt_centroid_y", "gt_side_lr", "gt_side_ud",
    "pred_centroid_x", "pred_centroid_y", "pred_side_lr", "pred_side_ud",
    "runtime_seconds",
]


def load_manifest(dataset_name):
    manifest_path = DATASET_ROOT / dataset_name / f"manifest_{dataset_name}.csv"
    with open(manifest_path, newline="") as f:
        return list(csv.DictReader(f))


def load_done_keys(results_path):
    if not results_path.exists():
        return set()
    with open(results_path, newline="") as f:
        rows = list(csv.DictReader(f))
    return {(r["sample_id"], r["region"]) for r in rows}


def append_results(results_path, rows):
    file_exists = results_path.exists()
    with open(results_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def mask_centroid_and_side(mask_bool):
    ys, xs = np.nonzero(mask_bool)
    if len(xs) == 0:
        return None
    cx, cy = float(xs.mean()), float(ys.mean())
    h, w = mask_bool.shape
    side_lr = "right" if cx > w / 2 else "left"
    side_ud = "upper" if cy < h / 2 else "lower"
    return cx, cy, side_lr, side_ud


def dice_iou(pred_bool, gt_bool):
    inter = np.logical_and(pred_bool, gt_bool).sum()
    union = np.logical_or(pred_bool, gt_bool).sum()
    pred_sum, gt_sum = pred_bool.sum(), gt_bool.sum()
    dice = (2.0 * inter / (pred_sum + gt_sum)) if (pred_sum + gt_sum) > 0 else 0.0
    iou = (inter / union) if union > 0 else 0.0
    return float(dice), float(iou)


def run_dataset(dataset_name, limit=LIMIT_PER_DATASET):
    """Runs BiomedParse over the first `limit` P_ALL rows of one dataset's
    manifest. Resumable: results are appended after every image, and rows
    already present in the output CSV (by sample_id+region) are skipped on
    a re-run, so a Colab disconnect loses at most the one image in progress.
    """
    region_map = REGION_PROMPTS[dataset_name]
    dataset_dir = DATASET_ROOT / dataset_name
    results_path = RESULTS_ROOT / f"{dataset_name}_results.csv"
    pred_masks_dir = RESULTS_ROOT / dataset_name / "pred_masks"
    pred_masks_dir.mkdir(parents=True, exist_ok=True)

    all_rows = load_manifest(dataset_name)
    p_all_rows = [r for r in all_rows if r["prompt_id"] == "P_ALL"][:limit]
    done = load_done_keys(results_path)

    for row in p_all_rows:
        sample_id = row["sample_id"]
        slugs = row["target_regions"].split(";")
        mask_rels = row["mask_paths"].split(";")
        remaining = [(s, m) for s, m in zip(slugs, mask_rels) if (sample_id, s) not in done]
        if not remaining:
            print(f"[{dataset_name}] {sample_id}: already done, skipping.")
            continue

        image_path = dataset_dir / row["image_path"]
        image = Image.open(image_path).convert("RGB")
        prompts = [region_map[s] for s, _ in remaining]

        t0 = time.time()
        pred_mask_prob = interactive_infer_image(model, image, prompts)  # (N, H, W)
        runtime = time.time() - t0
        per_region_runtime = runtime / len(remaining)

        result_rows = []
        for i, (slug, mask_rel) in enumerate(remaining):
            gt_mask = np.array(Image.open(dataset_dir / mask_rel).convert("L")) > 0
            pred_mask = pred_mask_prob[i] > 0.5

            dice, iou = dice_iou(pred_mask, gt_mask)
            gt_stats = mask_centroid_and_side(gt_mask)
            pred_stats = mask_centroid_and_side(pred_mask)

            pred_mask_path = pred_masks_dir / f"{sample_id}__{slug}.png"
            Image.fromarray((pred_mask * 255).astype(np.uint8)).save(pred_mask_path)

            result_rows.append({
                "dataset": dataset_name, "sample_id": sample_id, "prompt_id": "P_ALL",
                "region": slug, "prompt_text": region_map[slug],
                "image_path": row["image_path"], "gt_mask_path": mask_rel,
                "pred_mask_path": str(pred_mask_path.relative_to(RESULTS_ROOT)),
                "dice": round(dice, 4), "iou": round(iou, 4),
                "gt_centroid_x": gt_stats[0] if gt_stats else "",
                "gt_centroid_y": gt_stats[1] if gt_stats else "",
                "gt_side_lr": gt_stats[2] if gt_stats else "",
                "gt_side_ud": gt_stats[3] if gt_stats else "",
                "pred_centroid_x": pred_stats[0] if pred_stats else "",
                "pred_centroid_y": pred_stats[1] if pred_stats else "",
                "pred_side_lr": pred_stats[2] if pred_stats else "",
                "pred_side_ud": pred_stats[3] if pred_stats else "",
                "runtime_seconds": round(per_region_runtime, 1),
            })

        append_results(results_path, result_rows)
        print(
            f"[{dataset_name}] {sample_id}: {len(remaining)} region(s) in {runtime:.1f}s "
            f"-> {results_path}"
        )

    print(f"Done: {dataset_name} -> {results_path}")
    return results_path

In [ ]:
acdc_results = run_dataset("acdc")

In [ ]:
chaos_results = run_dataset("chaos")

In [ ]:
isic_results = run_dataset("isic_task2")

## Notes

- This runs all 20 `P_ALL` rows in each manifest (that's the full set -- ACDC,
  CHAOS, and ISIC Task 2 each have exactly 20). Already-done `(sample_id,
  region)` rows are skipped on a re-run, so an interrupted run just picks up
  where it left off rather than redoing work.
- Each output CSV (`acdc_results.csv`, `chaos_results.csv`,
  `isic_task2_results.csv` under `RESULTS_ROOT`) has one row **per region**,
  not per image -- e.g. one ACDC image with all 3 cardiac structures produces
  3 rows, all sharing the same `sample_id` and `prompt_id="P_ALL"`.
- Predicted masks are saved as PNGs under `RESULTS_ROOT/<dataset>/pred_masks/`,
  named `<sample_id>__<region>.png`.